<a href="https://colab.research.google.com/github/nivedita-rmsh/CLED/blob/main/XLM_R.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
import json
from collections import Counter

Verifying existence of MAVEN files

In [ ]:
maven_dir = '/content/drive/MyDrive/NLP_data'
for f in os.listdir(maven_dir):
    size = os.path.getsize(os.path.join(maven_dir, f)) / 1024
    print(f"{f:40s} {size:.1f} KB")

Load and Inspect Structure of JSON files

In [ ]:
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

train = load_jsonl(f'{maven_dir}/train.jsonl')

# Look at one document
doc = train[0]
print("Keys in a document:", list(doc.keys()))
print("Title:", doc['title'])
print("Number of sentences:", len(doc['content']))
print("Number of events:", len(doc['events']))

Looking into sentences with their event triggers highlighted.

In [ ]:
for doc in train[:3]:
    print(f"\n=== {doc['title']} ===")
    for event in doc['events'][:2]:
        etype = event['type']
        for mention in event['mention'][:1]:
            sid   = mention['sent_id']
            start, end = mention['offset']
            tokens = doc['content'][sid]['tokens']
            trigger = ' '.join(tokens[start:end])
            sentence = ' '.join(tokens)
            print(f"  Event type : {etype}")
            print(f"  Trigger    : '{trigger}'  (tokens {start}–{end})")
            print(f"  Sentence   : {sentence}\n")

Event Stats

In [ ]:
event_types = Counter()
total_mentions = 0

for doc in train:
    for event in doc['events']:
        event_types[event['type']] += len(event['mention'])
        total_mentions += len(event['mention'])

print(f"Total documents   : {len(train)}")
print(f"Total event types : {len(event_types)}")
print(f"Total mentions    : {total_mentions}")
print(f"\nTop 10 event types:")
for etype, count in event_types.most_common(10):
    print(f"  {etype:30s} {count}")

Convert to BIO format

In [ ]:
def doc_to_bio_examples(doc):
    examples = []
    trigger_map = {}
    for event in doc['events']:
        for mention in event['mention']:
            sid = mention['sent_id']
            trigger_map.setdefault(sid, []).append(
                (mention['offset'][0], mention['offset'][1], event['type'])
            )

    for sid, sent in enumerate(doc['content']):
        tokens = sent['tokens']
        labels = ['O'] * len(tokens)
        for start, end, etype in trigger_map.get(sid, []):
            for i in range(start, end):
                labels[i] = f"B-{etype}" if i == start else f"I-{etype}"
        examples.append({'tokens': tokens, 'labels': labels, 'doc_id': doc['id'], 'sent_id': sid})
    return examples

# Test it
sample = doc_to_bio_examples(train[0])
for ex in sample[:3]:
    pairs = list(zip(ex['tokens'], ex['labels']))
    non_o = [(t, l) for t, l in pairs if l != 'O']
    if non_o:
        print(ex['tokens'])
        print(ex['labels'])
        print()

Sanity Check

In [ ]:
def check_bio_integrity(examples):
    errors = 0
    for ex in examples:
        prev = 'O'
        for i, label in enumerate(ex['labels']):
            if label.startswith('I-'):
                etype = label[2:]
                if prev != f'B-{etype}' and prev != f'I-{etype}':
                    print(f"BIO error at token {i}: '{label}' follows '{prev}'")
                    print(f"  Tokens: {ex['tokens']}")
                    errors += 1
            prev = label
    print(f"\nTotal BIO errors: {errors}")

check_bio_integrity(sample)

Translation to FRENCH

In [ ]:
!pip install transformers sentencepiece sacremoses -q

In [ ]:
from transformers import MarianMTModel, MarianTokenizer
import torch
import json
from tqdm import tqdm

In [ ]:
model_name = "Helsinki-NLP/opus-mt-en-fr"
tokenizer  = MarianTokenizer.from_pretrained(model_name)
model      = MarianMTModel.from_pretrained(model_name)

print("Model loaded successfully")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model  = model.to(device)

def translate_batch(sentences, batch_size=32, max_length=128):
    """
    Translate a list of English strings to French.
    Returns a list of translated strings, same length as input.
    """
    results = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i : i + batch_size]
        safe_batch = [s if s.strip() else "." for s in batch]

        inputs = tokenizer(
            safe_batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_length
        ).to(device)

        with torch.no_grad():
            translated = model.generate(**inputs, max_length=max_length)

        decoded = tokenizer.batch_decode(translated, skip_special_tokens=True)
        results.extend(decoded)

    return results

In [ ]:

def translate_maven_split(input_path, output_path):
    """
    Reads a MAVEN .jsonl file, translates all sentence text to French,
    writes a new .jsonl with translated content alongside the original.
    """
    # Load all documents
    with open(input_path) as f:
        docs = [json.loads(line) for line in f]

    # Collect all sentences across all docs for batch translation
    # We track (doc_idx, sent_idx) so we can put them back
    all_sentences = []
    index_map     = []   # (doc_idx, sent_idx)

    for d_idx, doc in enumerate(docs):
        for s_idx, sent in enumerate(doc['content']):
            all_sentences.append(sent['sentence'])
            index_map.append((d_idx, s_idx))

    print(f"Translating {len(all_sentences)} sentences from {len(docs)} documents...")

    translated = []
    batch_size = 32
    for i in tqdm(range(0, len(all_sentences), batch_size)):
        batch = all_sentences[i : i + batch_size]
        translated.extend(translate_batch(batch, batch_size=batch_size))

    # Put translated sentences back into the document structure
    for (d_idx, s_idx), fr_sentence in zip(index_map, translated):
        docs[d_idx]['content'][s_idx]['sentence_fr'] = fr_sentence
        # Keep original sentence too — useful for debugging alignment

    with open(output_path, 'w') as f:
        for doc in docs:
            f.write(json.dumps(doc, ensure_ascii=False) + '\n')

    print(f"Done. Saved to {output_path}")

In [ ]:
maven_dir = '/content/drive/MyDrive/NLP_data'

translate_maven_split(
    f'{maven_dir}/train.jsonl',
    f'{maven_dir}/train_fr.jsonl'
)

translate_maven_split(
    f'{maven_dir}/valid.jsonl',
    f'{maven_dir}/valid_fr.jsonl'
)

translate_maven_split(
    f'{maven_dir}/test.jsonl',
    f'{maven_dir}/test_fr.jsonl'
)

In [ ]:
with open(f'{maven_dir}/train_fr.jsonl') as f:
    docs_fr = [json.loads(line) for line in f]

doc = docs_fr[0]
print(f"Document: {doc['title']}\n")

for sent in doc['content'][:4]:
    print(f"EN: {sent['sentence']}")
    print(f"FR: {sent['sentence_fr']}")
    print()

TOKENIZATION

In [ ]:
!pip install transformers -q

In [ ]:
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import XLMRobertaTokenizerFast
from transformers import XLMRobertaModel

In [ ]:
tokenizer = XLMRobertaTokenizerFast.from_pretrained("xlm-roberta-base")
print("Tokenizer loaded:", tokenizer.__class__.__name__)
print("Vocab size:", tokenizer.vocab_size)

Converting a MAVEN document into a list of sentence-level examples. Each example will have word-level tokens and BIO labels.

In [ ]:
maven_dir = '/content/drive/MyDrive/NLP_data'

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

def doc_to_bio_examples(doc, lang='en'):

    examples = []
    trigger_map = {}
    for event in doc['events']:
        for mention in event['mention']:
            sid = mention['sent_id']
            trigger_map.setdefault(sid, []).append((
                mention['offset'][0],
                mention['offset'][1],
                event['type']
            ))
    for sid, sent in enumerate(doc['content']):
        tokens = sent['tokens']
        labels = ['O'] * len(tokens)

        for start, end, etype in trigger_map.get(sid, []):
            for i in range(start, end):
                labels[i] = f"B-{etype}" if i == start else f"I-{etype}"

        examples.append({
            'tokens': tokens,
            'labels': labels,
            'sentence_fr': sent.get('sentence_fr', ''),
            'doc_id': doc['id'],
            'sent_id': sid
        })
    return examples

train_docs = load_jsonl(f'{maven_dir}/train_fr.jsonl')

all_examples = []
for doc in train_docs:
    all_examples.extend(doc_to_bio_examples(doc))

print(f"Total sentence examples: {len(all_examples)}")

Build the label vocabulary

In [ ]:
all_label_types = set()
for ex in all_examples:
    for label in ex['labels']:
        all_label_types.add(label)

sorted_labels = ['O'] + sorted([l for l in all_label_types if l != 'O'])
label2id = {l: i for i, l in enumerate(sorted_labels)}
id2label  = {i: l for l, i in label2id.items()}

print(f"Total unique labels: {len(label2id)}")
print(f"Sample labels: {sorted_labels[:10]}")

word_ids() maps each subword token back to its source word index. We use this to copy the right label to every subword piece of a word.

The core alignment function

In [ ]:
def tokenize_and_align(example, max_length=128):
    encoding = tokenizer(
        example['tokens'],
        is_split_into_words=True,
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    word_ids = encoding.word_ids()
    word_labels = example['labels']
    aligned_labels = []

    for word_idx in word_ids:
        if word_idx is None:
            aligned_labels.append(-100)
        else:
            label_str = word_labels[word_idx]
            aligned_labels.append(label2id[label_str])

    return {
        'input_ids':      encoding['input_ids'].squeeze(),
        'attention_mask': encoding['attention_mask'].squeeze(),
        'labels':         aligned_labels,
        'lang':           'en'
    }

Verify alignment visually

In [ ]:
def inspect_alignment(example, max_length=128):
    encoding = tokenizer(
        example['tokens'],
        is_split_into_words=True,
        max_length=max_length,
        truncation=True
    )

    word_ids   = encoding.word_ids()
    tokens     = tokenizer.convert_ids_to_tokens(encoding['input_ids'])
    word_labels = example['labels']

    print(f"\n{'Subword token':<20} {'Word idx':<12} {'Label'}")
    print("-" * 45)
    for token, word_idx in zip(tokens, word_ids):
        if word_idx is None:
            label = '[SPECIAL]'
        else:
            label = word_labels[word_idx]
        marker = " ← trigger" if label not in ('O', '[SPECIAL]') else ""
        print(f"{token:<20} {str(word_idx):<12} {label}{marker}")

example_with_events = next(
    ex for ex in all_examples
    if any(l != 'O' for l in ex['labels'])
)
inspect_alignment(example_with_events)

Tokenize the French sentences for alignment

In [ ]:
def tokenize_french(example, max_length=128):
    if not example['sentence_fr'].strip():
        return None

    encoding = tokenizer(
        example['sentence_fr'],
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

    return {
        'input_ids':      encoding['input_ids'].squeeze(),
        'attention_mask': encoding['attention_mask'].squeeze(),
        'lang':           'fr'
    }

sample_fr = tokenize_french(example_with_events)
fr_tokens  = tokenizer.convert_ids_to_tokens(
    sample_fr['input_ids']
)
print("French subword tokens:")
print([t for t in fr_tokens if t not in ('<pad>', '<s>', '</s>')])

Tokenize the full dataset and build a DataLoader

In [ ]:
class MAVENDataset(Dataset):
    def __init__(self, examples, max_length=128):
        self.items = []
        skipped = 0
        for ex in examples:
            en = tokenize_and_align(ex, max_length)
            fr = tokenize_french(ex, max_length)
            if fr is None:
                skipped += 1
                continue
            self.items.append({
                # English — used for event detection loss
                'en_input_ids':      en['input_ids'],
                'en_attention_mask': en['attention_mask'],
                'labels':            torch.tensor(en['labels']),
                # French — used for alignment loss
                'fr_input_ids':      fr['input_ids'],
                'fr_attention_mask': fr['attention_mask'],
            })
        print(f"Built dataset: {len(self.items)} examples ({skipped} skipped)")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        return self.items[idx]

train_dataset = MAVENDataset(all_examples[:500])
train_loader  = DataLoader(train_dataset, batch_size=16, shuffle=True)

batch = next(iter(train_loader))
print("\nBatch shapes:")
for k, v in batch.items():
    print(f"  {k:<25} {tuple(v.shape)}")

BUILD DATALOADER

In [ ]:
batch = next(iter(train_loader))

print("=" * 45)
print("BATCH SHAPES")
print("=" * 45)
for key, tensor in batch.items():
    print(f"  {key:<25} {tuple(tensor.shape)}")

In [ ]:
def verify_label_integrity(batch, num_samples=3):
    labels       = batch['labels']
    input_ids    = batch['en_input_ids']
    attn_mask    = batch['en_attention_mask']

    print("=" * 55)
    print("LABEL INTEGRITY CHECK")
    print("=" * 55)

    total_errors = 0

    for i in range(min(num_samples, labels.shape[0])):
        seq_labels = labels[i]
        seq_mask   = attn_mask[i]
        seq_ids    = input_ids[i]

        # Find positions where label is -100
        minus100_positions = (seq_labels == -100).nonzero(as_tuple=True)[0]

        # Find positions where attention mask is 1 (real tokens)
        real_token_positions = (seq_mask == 1).nonzero(as_tuple=True)[0]

        # BAD: -100 appearing on a real non-special token
        # Special tokens: XLM-R uses id=0 for pad, 0=<s>, 2=</s>
        special_ids = {0, 1, 2}   # <pad>, <s>, </s>
        errors = []
        for pos in minus100_positions:
            pos = pos.item()
            token_id = seq_ids[pos].item()
            is_real  = seq_mask[pos].item() == 1
            is_special = token_id in special_ids
            if is_real and not is_special:
                errors.append(pos)
                total_errors += 1

        # Stats for this sequence
        n_minus100  = (seq_labels == -100).sum().item()
        n_real      = seq_mask.sum().item()
        n_pad       = (seq_mask == 0).sum().item()
        n_event     = ((seq_labels > 0) & (seq_labels != -100)).sum().item()

        print(f"\nSample {i+1}:")
        print(f"  Total tokens (seq len)  : 128")
        print(f"  Real tokens (mask=1)    : {n_real}")
        print(f"  Padding tokens (mask=0) : {n_pad}")
        print(f"  Labels == -100          : {n_minus100}  (should be CLS + SEP + padding only)")
        print(f"  Event trigger labels    : {n_event}  (non-O, non -100)")
        print(f"  Leak errors             : {len(errors)}" + (" ✓" if not errors else " ✗ BAD"))

    print(f"\n{'=' * 55}")
    print(f"Total -100 leak errors across {num_samples} samples: {total_errors}")
    if total_errors == 0:
        print("  All clean — no leaks ✓")
    else:
        print("  Leaks detected — check your alignment function ✗")
    print("=" * 55)

verify_label_integrity(batch)

Visually decode one sequence end to end

In [ ]:
def decode_one_sequence(batch, sample_idx=0):
    ids    = batch['en_input_ids'][sample_idx]
    mask   = batch['en_attention_mask'][sample_idx]
    labels = batch['labels'][sample_idx]

    tokens = tokenizer.convert_ids_to_tokens(ids)

    print("=" * 65)
    print(f"DECODED SEQUENCE (sample {sample_idx + 1})")
    print("=" * 65)
    print(f"{'Token':<20} {'Mask':<8} {'Label ID':<12} {'Label'}")
    print("-" * 65)

    for token, m, label_id in zip(tokens, mask, labels):
        m        = m.item()
        label_id = label_id.item()

        if label_id == -100:
            label_str = '[SPECIAL/PAD]'
        else:
            label_str = id2label[label_id]

        # Skip padding for readability
        if token == '<pad>':
            continue

        marker = " ◄" if label_str not in ('O', '[SPECIAL/PAD]') else ""
        print(f"{token:<20} {m:<8} {str(label_id):<12} {label_str}{marker}")

decode_one_sequence(batch, sample_idx=0)

Check label distribution across the full batch

In [ ]:
def batch_label_stats(batch):
    labels = batch['labels']

    flat = labels.view(-1)
    real = flat[flat != -100]

    n_O      = (real == 0).sum().item()
    n_event  = (real != 0).sum().item()
    total    = real.shape[0]

    print("=" * 45)
    print("LABEL DISTRIBUTION (this batch)")
    print("=" * 45)
    print(f"  Total real label positions : {total}")
    print(f"  O (non-event)              : {n_O}  ({100*n_O/total:.1f}%)")
    print(f"  Event triggers             : {n_event}  ({100*n_event/total:.1f}%)")
    print()

    from collections import Counter
    event_counts = Counter()
    for label_id in real.tolist():
        if label_id != 0:
            event_counts[id2label[label_id]] += 1

    print("  Top event types in this batch:")
    for label, count in event_counts.most_common(8):
        print(f"    {label:<30} {count}")

batch_label_stats(batch)

In [ ]:
batch = next(iter(train_loader))

verify_label_integrity(batch, num_samples=3)
print()
decode_one_sequence(batch, sample_idx=0)
print()
batch_label_stats(batch)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

CLED model

In [ ]:
class CLEDNetwork(nn.Module):
    def __init__(self, num_labels, hidden_size=768, proj_size=256, dropout=0.1):
        super().__init__()
        self.encoder = XLMRobertaModel.from_pretrained("xlm-roberta-base")

        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_size, num_labels)

        self.projection = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.GELU(),
            nn.Linear(hidden_size, proj_size)
        )

    def forward(self, input_ids, attention_mask, return_projection=False):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        sequence_output = outputs.last_hidden_state
        cls_output      = sequence_output[:, 0, :]

        logits = self.classifier(self.dropout(sequence_output))

        if return_projection:
            # L2-normalized projection for alignment loss
            proj = self.projection(cls_output)
            proj = nn.functional.normalize(proj, dim=-1)
            return logits, proj

        return logits

num_labels = len(label2id)
model      = CLEDNetwork(num_labels=num_labels).to(device)

print(f"Model built — num_labels: {num_labels}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

Freeze the bottom 6 encoder layers

In [ ]:
def freeze_bottom_layers(model, n_layers=6):
    # Freeze embeddings
    for param in model.encoder.embeddings.parameters():
        param.requires_grad = False

    # Freeze first n_layers transformer layers
    for i in range(n_layers):
        for param in model.encoder.encoder.layer[i].parameters():
            param.requires_grad = False

    # Report what's frozen vs trainable
    frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = frozen + trainable

    print(f"Frozen parameters    : {frozen:>12,}  ({100*frozen/total:.1f}%)")
    print(f"Trainable parameters : {trainable:>12,}  ({100*trainable/total:.1f}%)")
    print(f"Total parameters     : {total:>12,}")

freeze_bottom_layers(model, n_layers=6)

In [ ]:
def print_layer_status(model):
    print(f"\n{'Layer':<55} {'Trainable'}")
    print("-" * 65)

    emb_trainable = any(p.requires_grad
                        for p in model.encoder.embeddings.parameters())
    print(f"{'encoder.embeddings':<55} {emb_trainable}")

    # Transformer layers
    for i, layer in enumerate(model.encoder.encoder.layer):
        trainable = any(p.requires_grad for p in layer.parameters())
        status    = "✓ trainable" if trainable else "✗ frozen"
        print(f"  {'encoder.layer[' + str(i) + ']':<51} {status}")

    # Heads
    for name, module in [("classifier (event head)", model.classifier),
                          ("projection (alignment)", model.projection)]:
        trainable = any(p.requires_grad for p in module.parameters())
        print(f"{'  ' + name:<55} {'✓ trainable' if trainable else '✗ frozen'}")

print_layer_status(model)

In [ ]:
def verify_forward_pass(model, batch):
    model.eval()
    with torch.no_grad():
        # Move batch to device
        input_ids   = batch['en_input_ids'].to(device)
        attn_mask   = batch['en_attention_mask'].to(device)
        labels      = batch['labels'].to(device)

        # Forward pass — with projection for alignment
        logits, proj = model(
            input_ids=input_ids,
            attention_mask=attn_mask,
            return_projection=True
        )

    print("=" * 50)
    print("FORWARD PASS VERIFICATION")
    print("=" * 50)
    print(f"  input_ids shape    : {tuple(input_ids.shape)}")
    print(f"  logits shape       : {tuple(logits.shape)}")
    print(f"  projection shape   : {tuple(proj.shape)}")
    print()
    print(f"  Expected logits    : (batch, 128, {len(label2id)})")
    print(f"  Expected projection: (batch, 256)")
    print()

    B, S, L = logits.shape
    assert S == 128,         f"Seq length mismatch: {S}"
    assert L == len(label2id), f"Label count mismatch: {L}"
    assert proj.shape[1] == 256, f"Projection dim mismatch: {proj.shape}"

    norms = proj.norm(dim=-1)
    print(f"  Projection norms   : min={norms.min():.4f}  max={norms.max():.4f}  (should be ~1.0)")
    print()
    print("  All checks passed ✓")

batch = next(iter(train_loader))
verify_forward_pass(model, batch)

Confirm the loss computes correctly

In [ ]:
def verify_loss(model, batch):
    model.train()

    input_ids = batch['en_input_ids'].to(device)
    attn_mask = batch['en_attention_mask'].to(device)
    labels    = batch['labels'].to(device)

    logits    = model(input_ids=input_ids, attention_mask=attn_mask)

    loss_fn   = nn.CrossEntropyLoss(ignore_index=-100)
    loss      = loss_fn(logits.view(-1, num_labels), labels.view(-1))

    print("=" * 50)
    print("LOSS VERIFICATION")
    print("=" * 50)
    print(f"  Event detection loss : {loss.item():.4f}")
    print(f"  (Expected ~log({num_labels}) = {torch.log(torch.tensor(float(num_labels))):.2f} for random init)")

    assert not torch.isnan(loss), "Loss is NaN — something is wrong"
    assert not torch.isinf(loss), "Loss is Inf — something is wrong"
    print("  No NaN or Inf ✓")

verify_loss(model, batch)